# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — the same slice, the same split, the same frozen baseline

Nothing about the data is re-chosen this week. Same lane constants as the w03 contract, same two
queries, same grouped split, same metric: **features from `month=2026-03`, label from
`month=2026-04`, decision moment 2026-03-31, precision@50 on held-out clients.**

w04 ended by freezing one number: **precision@50 = 90.0%** for the hand-written rule, measured on
9 held-out clients. This notebook has to beat that number *on those same rows*, or say plainly
that it didn't. So section 0 rebuilds the slice, re-derives the split, and recomputes the frozen
baseline from the rule's own code — then checks all three against the committed receipts before a
single model is fitted. If any of them had drifted, every comparison below would be a model
measured against a different week's work.

In [1]:
import importlib.util, subprocess, sys

for pkg in ("duckdb", "huggingface_hub"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import json, os, pathlib
import duckdb, numpy as np, pandas as pd, sklearn

pd.set_option("display.width", 170)
SEED = 42          # same seed as w03 and w04 — same split, same numbers on a rerun

# Tree ensembles can move a point or two between library versions, so the versions are part of
# the result, not trivia.
print(f"python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scikit-learn {sklearn.__version__} | duckdb {duckdb.__version__}")


def load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "environment variable"
    for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        env_file = parent / ".env"
        if env_file.exists():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith("HF_TOKEN="):
                    return line.split("=", 1)[1].strip().strip("\"'"), "local .env (gitignored)"
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN"), "Colab Secret"
    except Exception:
        pass
    raise RuntimeError("No HF_TOKEN found. Set a Colab Secret named HF_TOKEN (read token).")


HF_TOKEN, token_source = load_hf_token()
print(f"HF read token loaded from: {token_source}")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"


def fact(month):
    """One month partition of fact_content_daily_performance."""
    return f"read_parquet('{REL}/fact_content_daily_performance/month={month}/*.parquet')"


# --- lane constants: copied unchanged from the w03 contract / w04 baseline ---
FEATURE_MONTH = "2026-03"          # features are measured here
LABEL_MONTH   = "2026-04"          # strictly after the decision moment
DECISION_DATE = "2026-03-31"       # the moment the editor scores the queue
MID_MONTH     = "2026-03-17"       # splits the feature month into two halves
MIN_HISTORY   = "2026-01-01"       # client must have GSC history starting on/before this
MIN_IMP_MAR   = 100                # a page needs measurable March search demand
DECLINE_DROP  = 0.20               # "declining" = April impressions >20% below March
TOP_K         = 50                 # editorial capacity: the queue is 50 pages long

CONTRACT_FEATURES = ["log_imp_mar", "active_days", "pos_mar", "ctr_mar", "momentum_in_month"]

receipts = {"lane": "bleed_tracker", "seed": SEED, "top_k": TOP_K,
            "feature_month": FEATURE_MONTH, "label_month": LABEL_MONTH,
            "sklearn_version": sklearn.__version__,
            "contract_features": CONTRACT_FEATURES}


def precision_at_k(df, score_col, label_col="label_declining", k=TOP_K):
    """w04's metric, unchanged: of the top k rows this score ranks highest, how many declined?"""
    return float(df.nlargest(k, score_col)[label_col].mean())


def precision_at_k_fair(df, score_col, label_col="label_declining", k=TOP_K):
    """The same metric, but ties are not allowed to choose the queue.

    `nlargest` breaks ties by row order, which is scan order, which is nobody's ranking. When
    many pages share the score sitting at rank k, this reports the *expected* precision over the
    tied block instead of the one arrangement pandas happened to hand back.
    """
    s = df[score_col].to_numpy()
    y = df[label_col].to_numpy()
    cut = np.sort(s)[::-1][k - 1]
    above, tied = s > cut, s == cut
    slots = k - int(above.sum())          # how many of the k places the tied block has to fill
    fair = (y[above].sum() + slots * y[tied].mean()) / k
    return {"as_listed": float(df.nlargest(k, score_col)[label_col].mean()),
            "tie_fair": float(fair), "tied_at_cut": int(tied.sum()), "slots_from_ties": int(slots)}


print(f"\ndecision moment {DECISION_DATE} | features from {FEATURE_MONTH} | label from {LABEL_MONTH}")
print(f"metric: precision@{TOP_K} on held-out clients | number to beat: w04's frozen 90.0%")

python 3.12.4 | numpy 1.26.4 | pandas 2.3.3 | scikit-learn 1.9.0 | duckdb 1.5.4
HF read token loaded from: environment variable

decision moment 2026-03-31 | features from 2026-03 | label from 2026-04
metric: precision@50 on held-out clients | number to beat: w04's frozen 90.0%


### 0b. Rebuild the contract slice

The same two queries as w03 and w04, unchanged. One row = one page-month decision row:
`gsc_data_available IS TRUE`, clients whose GSC history starts before 2026-01-01, pages with at
least 100 March impressions. The five contract features are rebuilt from the same columns, and
the label is still the only thing measured after the decision moment.

**One line is new: `ORDER BY content_hash_id`.** A parallel `GROUP BY`
returns its rows in whatever order the threads finish in — I checked, and three runs of this exact
query gave three different row orders. Row order is not supposed to matter, but it does: a random
forest's bootstrap draws depend on it, and so does which page `nlargest` puts in a queue when
scores tie. Without this line the notebook prints slightly different numbers every run, which
would make "fix your seeds" a decoration. Section 3 shows what that costs when it is left
unpinned.

In [2]:
march = con.sql(f"""
    SELECT f.client_hash_id,
           f.content_hash_id,
           SUM(f.gsc_impressions)                                          AS imp_mar,
           SUM(f.gsc_clicks)                                               AS clk_mar,
           COUNT(*) FILTER (f.gsc_impressions > 0)                         AS active_days,
           SUM(f.gsc_sum_position)                                         AS sum_pos_mar,
           MAX(f.gsc_impressions)                                          AS max_day_imp,
           SUM(f.gsc_impressions) FILTER (f.report_date >= DATE '{MID_MONTH}') AS imp_late,
           SUM(f.gsc_impressions) FILTER (f.report_date <  DATE '{MID_MONTH}') AS imp_early
    FROM {fact(FEATURE_MONTH)} f
    JOIN {DIM_CLIENTS} c USING (client_hash_id)
    WHERE f.gsc_data_available IS TRUE
      AND c.gsc_data_start <= DATE '{MIN_HISTORY}'
    GROUP BY 1, 2
    HAVING SUM(f.gsc_impressions) >= {MIN_IMP_MAR}
    ORDER BY f.content_hash_id
""").df()

april = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr
    FROM {fact(LABEL_MONTH)}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1
""").df()

assert not march.content_hash_id.duplicated().any(), "content_hash_id is not unique in-month"
assert not april.content_hash_id.duplicated().any(), "label key is not unique"

frame = march.merge(april, on="content_hash_id", how="left")
assert len(frame) == len(march), "the label join changed the row count"
frame["imp_apr"] = frame.imp_apr.fillna(0)

# --- the five contract features, rebuilt exactly as in w03 -------------------
frame["pos_mar"] = frame.sum_pos_mar / frame.imp_mar
frame["ctr_mar"] = frame.clk_mar / frame.imp_mar
frame["momentum_in_month"] = frame.imp_late / frame.imp_early.replace(0, np.nan)
frame["has_first_half_traffic"] = frame.momentum_in_month.notna().astype(int)   # w03 context, not a feature
frame["momentum_in_month"] = frame.momentum_in_month.fillna(1.0)
frame["log_imp_mar"] = np.log1p(frame.imp_mar)
frame["spike_share"] = frame.max_day_imp / frame.imp_mar                        # w04 review column

# --- the label (the only column measured after the decision moment) ----------
frame["label_declining"] = (frame.imp_apr < (1 - DECLINE_DROP) * frame.imp_mar).astype(int)

print(f"slice: {len(frame):,} page-month rows | {frame.client_hash_id.nunique()} clients")
print(f"base rate (share declining next month): {frame.label_declining.mean():.2%}")

w03_path = pathlib.Path("../outputs/w03_data_contract_receipts.json")
if w03_path.exists():
    prev = json.loads(w03_path.read_text(encoding="utf-8"))
    assert len(frame) == prev["frame_rows"], "slice size drifted from the w03 contract"
    assert round(float(frame.label_declining.mean()), 4) == prev["label_base_rate"], "base rate drifted"
    print(f"cross-check vs w03 receipts: {prev['frame_rows']:,} rows, base rate "
          f"{prev['label_base_rate']:.2%} — match.")
else:
    print("w03 receipts not found next to this notebook — skipping the cross-check.")

receipts["slice_rows"] = int(len(frame))
receipts["slice_base_rate"] = round(float(frame.label_declining.mean()), 4)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

slice: 85,453 page-month rows | 27 clients
base rate (share declining next month): 53.78%
cross-check vs w03 receipts: 85,453 rows, base rate 53.78% — match.


### 0c. The frozen baseline, recomputed here

The baseline appears **in the same table as the
model, computed in the same notebook run** — a number copied out of last week's markdown is a
claim, not a comparison. So w04's rule is pasted back in verbatim (constants and all), its one
fitted quantity is re-derived on the same training clients, and it is scored on the same held-out
clients. The assertion at the bottom is the real check: it fails loudly if this notebook has
quietly become a different experiment.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
train_idx, test_idx = next(splitter.split(frame, groups=frame.client_hash_id))
frame["split"] = "train"
frame.loc[frame.index[test_idx], "split"] = "test"

# ---------------- w04's rule, frozen: nothing below this line was retuned ----------------
POS_BANDS, POS_NAMES = [0, 3, 10, 20, np.inf], ["1-3", "4-10", "11-20", "21+"]
SLIP_GATE, POS_REACH, W_UNDERCLICK, W_REACHABLE = 0.8, 20, 1.0, 0.5


def band_median_ctr(df):
    """The rule's only fitted quantity — learned on training clients only."""
    return (df.assign(pos_band=pd.cut(df.pos_mar, POS_BANDS, labels=POS_NAMES))
              .groupby("pos_band", observed=True)["ctr_mar"].median())


def score_pages(df, band_med_ctr):
    """w04's baseline action score, unchanged. Inputs: imp_mar, momentum_in_month, ctr_mar, pos_mar."""
    band       = pd.cut(df.pos_mar, POS_BANDS, labels=POS_NAMES)
    severity   = (1 - df.momentum_in_month / SLIP_GATE).clip(lower=0)
    underclick = (df.ctr_mar < band.map(band_med_ctr).astype(float)).astype(int)
    reachable  = ((df.pos_mar >= 1) & (df.pos_mar <= POS_REACH)).astype(int)
    return severity * np.log1p(df.imp_mar) * (1 + W_UNDERCLICK * underclick) * (1 + W_REACHABLE * reachable)
# ----------------------------------------------------------------------------------------

BAND_MED_CTR = band_median_ctr(frame[frame.split == "train"])
frame["action_score"] = score_pages(frame, BAND_MED_CTR)
train, test = frame[frame.split == "train"].copy(), frame[frame.split == "test"].copy()

print(f"train {len(train):,} rows / {train.client_hash_id.nunique()} clients — model selection happens here")
print(f"test  {len(test):,} rows / {test.client_hash_id.nunique()} clients — opened once, in section 3")
print(f"train base rate {train.label_declining.mean():.2%} | test base rate {test.label_declining.mean():.2%}")

BASELINE_P50 = precision_at_k(test, "action_score")
print(f"\nfrozen baseline recomputed in this run: precision@{TOP_K} = {BASELINE_P50:.1%}")

w04_path = pathlib.Path("../outputs/w04_baseline_score_receipts.json")
if w04_path.exists():
    w04 = json.loads(w04_path.read_text(encoding="utf-8"))
    assert w04["frozen"] is True, "w04 did not declare its baseline frozen"
    assert round(BASELINE_P50, 4) == w04["baseline_precision_at_50_test"], "the baseline moved"
    assert len(test) == w04["test_rows"], "the held-out rows are not last week's held-out rows"
    assert round(float(test.label_declining.mean()), 4) == w04["test_base_rate"], "test base rate moved"
    print(f"matches the committed w04 receipt ({w04['baseline_precision_at_50_test']:.1%}) on the same "
          f"{w04['test_rows']:,} held-out rows. The comparison is legitimate.")
else:
    print("w04 receipts not found next to this notebook — skipping the frozen-baseline cross-check.")

receipts["baseline_precision_at_50_test"] = round(BASELINE_P50, 4)
receipts["test_rows"], receipts["test_clients"] = int(len(test)), int(test.client_hash_id.nunique())
receipts["test_base_rate"] = round(float(test.label_declining.mean()), 4)

train 59,682 rows / 18 clients — model selection happens here
test  25,771 rows / 9 clients — opened once, in section 3
train base rate 55.69% | test base rate 49.36%

frozen baseline recomputed in this run: precision@50 = 90.0%
matches the committed w04 receipt (90.0%) on the same 25,771 held-out rows. The comparison is legitimate.


## 1. Method choice and why

**The question decides the method, not the other way round.** My lane's question — *which pages
does the editor open first?* — is a **ranking** question, and w02 fixed the shape of the answer:
a classifier trained on an observed label, whose predicted probability is the priority score, read
at **precision@50** because editorial capacity is 50 pages. That rules three things off my menu
immediately:

- **Clustering (K-Means)** answers "what groups exist here". I already have a named outcome
  measured in April, so I would be throwing away the label to rediscover a weaker version of it.
- **PCA** compresses correlated columns. I have five features and a reviewer who has to be able
  to argue with the score; trading names for components costs me the thing the queue needs most.
- **Signal analysis on its own** is what w04 already did (three bucket tables with verdicts). It
  told me *which* signals point the right way; it cannot hand an editor an ordered list.

So the menu is the four supervised methods, and I fit them in order of how much I would have to
explain to trust them:

| Candidate | Why it is on the list | What it costs |
|---|---|---|
| **Logistic Regression** | readable — one coefficient per feature, sign and size arguable | assumes the log-odds move in a straight line with each feature |
| **Decision Tree (depth 3)** | printable — the whole model fits on one screen, and it is the closest thing to my hand-written rule | coarse: a depth-3 tree has 8 leaves, so it can only produce 8 distinct scores |
| **Random Forest** | many trees vote, so it captures interactions the rule states by hand (slide × underclicking) | not readable directly; needs permutation importance to explain |
| **Gradient Boosting** | usually the strongest tabular learner; w03 used it for the leakage demo | most capacity, most chances to fit this one month pair rather than the pattern |

**Simplicity is the tie-breaker, not an afterthought.** If two of these land within a page or two
of each other at precision@50, I take the one I can explain, because the deliverable is a queue an
editor has to trust. Complexity has to *earn* its place in the comparison table.

**Everything below is decided on training clients only.** The candidates are compared with grouped
5-fold cross-validation *inside* the training clients — the held-out clients stay sealed until
section 3, and I pick one model before I open them.

In [4]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

FEATURES = CONTRACT_FEATURES          # the five w03 contract features, nothing added
gkf = GroupKFold(n_splits=5)


def make_candidates():
    """Fresh, unfitted estimators — reasonable defaults, no tuning yet."""
    return {
        "logistic regression":     make_pipeline(StandardScaler(),
                                                 LogisticRegression(max_iter=2000, random_state=SEED)),
        "decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=SEED),
        "random forest":           RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                                          n_jobs=-1, random_state=SEED),
        "gradient boosting":       HistGradientBoostingClassifier(max_iter=150, random_state=SEED),
    }


def grouped_cv(make_model, features=FEATURES, frame_in=None):
    """Fit on 4 folds of training clients, score the 5th. Returns per-fold precision@50 and AUC."""
    src = train if frame_in is None else frame_in
    out = []
    for fold, (a, b) in enumerate(gkf.split(src, src.label_declining, groups=src.client_hash_id)):
        A, B = src.iloc[a], src.iloc[b]
        model = make_model()
        model.fit(A[features], A.label_declining)
        risk = model.predict_proba(B[features])[:, 1]
        B = B.assign(risk=risk)
        out.append({"fold": fold, "val_rows": len(B), "val_clients": B.client_hash_id.nunique(),
                    "val_base": B.label_declining.mean(),
                    "p_at_50": precision_at_k_fair(B, "risk")["tie_fair"],
                    "rule_p_at_50": precision_at_k_fair(B, "action_score")["tie_fair"],
                    "auc": roc_auc_score(B.label_declining, risk)})
    return pd.DataFrame(out)


rows = []
for name, model in make_candidates().items():
    folds = grouped_cv(lambda n=name: make_candidates()[n])
    rows.append({"candidate": name, "p50_mean": folds.p_at_50.mean(), "p50_worst_fold": folds.p_at_50.min(),
                 "p50_best_fold": folds.p_at_50.max(), "auc_mean": folds.auc.mean(),
                 "beats_rule_in": int((folds.p_at_50 > folds.rule_p_at_50).sum())})

rule_folds = folds[["fold", "val_rows", "val_clients", "val_base", "rule_p_at_50"]]
shortlist = pd.DataFrame(rows)
print("grouped 5-fold CV inside the 18 training clients — held-out clients untouched")
print("\nthe folds themselves (the panel is concentrated, so the folds are very uneven):")
print(rule_folds.round(3).to_string(index=False))
print("\ncandidate comparison (precision@50 is tie-fair; 'beats_rule_in' counts folds out of 5)")
print(shortlist.round(3).to_string(index=False))
print(f"\nthe rule on the same five folds: mean {rule_folds.rule_p_at_50.mean():.3f}, "
      f"worst {rule_folds.rule_p_at_50.min():.3f}")

receipts["cv_shortlist"] = {r["candidate"]: round(r["p50_mean"], 4) for r in rows}
receipts["cv_rule_mean"] = round(float(rule_folds.rule_p_at_50.mean()), 4)

grouped 5-fold CV inside the 18 training clients — held-out clients untouched

the folds themselves (the panel is concentrated, so the folds are very uneven):
 fold  val_rows  val_clients  val_base  rule_p_at_50
    0     17313            1     0.726          1.00
    1     12764            1     0.473          0.94
    2      9659           10     0.389          0.88
    3      9932            3     0.738          0.92
    4     10014            3     0.355          0.72

candidate comparison (precision@50 is tie-fair; 'beats_rule_in' counts folds out of 5)
              candidate  p50_mean  p50_worst_fold  p50_best_fold  auc_mean  beats_rule_in
    logistic regression     0.720           0.380           0.92     0.611              1
decision tree (depth 3)     0.807           0.624           0.92     0.673              0
          random forest     0.904           0.780           0.96     0.707              2
      gradient boosting     0.889           0.744           0.98     0.710 

**What the shortlist says — and the first uncomfortable number.**

- **Logistic Regression is out** (0.72 mean, and one fold at 0.38). Section 4 shows why in one
  line: `momentum_in_month` runs from 0.001 to 4,091, so once it is standardised the whole
  informative range — pages between "collapsed" and "flat" — is squeezed into a sliver of the
  axis. A straight line in that space cannot say "0.1 is much worse than 0.9". Trees split on
  *order*, not distance, so the same column works for them untouched.
- **The depth-3 tree is weak here for a structural reason**, not a tuning one: 8 leaves means 8
  possible scores, so it cannot order 25,771 pages — it can only sort them into 8 heaps. Section 3
  shows what that does to a top-50 queue, and it is the most instructive result in this notebook.
- **The two ensembles lead**, and they lead on AUC (0.707, 0.710) by more than they lead at the
  head of the queue.
- **The rule is not behind them.** On the same five folds the frozen rule averages **0.892** — the
  untuned forest is 1.2pp above it, boosting 0.3pp below, and each of them beats the rule in only
  2 folds out of 5. Before any tuning the honest reading is: *the model has not yet earned its
  place; it has drawn level with a rule I wrote by hand.*

The folds are wildly uneven — one fold is a single client with 17,313 pages, another holds ten
small clients — because `GroupKFold` balances rows, and my panel is concentrated (w03 measured
the top 3 clients at 60.5% of rows). That unevenness is not a bug to fix; it is what "a new
client arrives" actually looks like here, and it is why I read the *worst* fold as well as the mean.

In [5]:
sweep = []
for leaf in (20, 100, 400, 1600):
    for feats, tag in [(FEATURES, "5 contract features"),
                       (FEATURES + ["has_first_half_traffic"], "+ momentum-missing flag")]:
        folds = grouped_cv(lambda l=leaf: RandomForestClassifier(n_estimators=300, min_samples_leaf=l,
                                                                 n_jobs=-1, random_state=SEED), feats)
        sweep.append({"candidate": f"random forest, min_samples_leaf={leaf}", "features": tag,
                      "p50_mean": folds.p_at_50.mean(), "p50_worst_fold": folds.p_at_50.min(),
                      "auc_mean": folds.auc.mean()})

for iters, lr, leaf in ((150, 0.10, 20), (150, 0.05, 200), (300, 0.03, 500)):
    folds = grouped_cv(lambda i=iters, r=lr, l=leaf: HistGradientBoostingClassifier(
        max_iter=i, learning_rate=r, min_samples_leaf=l, random_state=SEED))
    sweep.append({"candidate": f"gradient boosting, iter={iters} lr={lr} leaf={leaf}",
                  "features": "5 contract features", "p50_mean": folds.p_at_50.mean(),
                  "p50_worst_fold": folds.p_at_50.min(), "auc_mean": folds.auc.mean()})

sweep = pd.DataFrame(sweep)
print("regularisation sweep, still inside the training clients only")
print(sweep.round(3).to_string(index=False))
print(f"\nfor reference on the same folds — the frozen rule: mean {rule_folds.rule_p_at_50.mean():.3f}, "
      f"worst {rule_folds.rule_p_at_50.min():.3f}")

# The model I commit to, chosen here, before the held-out clients are opened.
CHOSEN_NAME = "random forest (chosen)"
CHOSEN_PARAMS = {"n_estimators": 300, "min_samples_leaf": 400, "n_jobs": -1, "random_state": SEED}
make_chosen = lambda: RandomForestClassifier(**CHOSEN_PARAMS)

# Boosting stays in the comparison as a second opinion, in its best CV setting above.
BOOSTING_PARAMS = {"max_iter": 150, "random_state": SEED}
make_boosting = lambda: HistGradientBoostingClassifier(**BOOSTING_PARAMS)

print(f"\nCHOSEN: random forest, {CHOSEN_PARAMS} on the 5 contract features. Frozen from here.")

receipts["cv_sweep"] = {f"{r.candidate} | {r.features}": round(r.p50_mean, 4) for r in sweep.itertuples()}
receipts["chosen_model"] = "RandomForestClassifier"
receipts["chosen_params"] = {k: v for k, v in CHOSEN_PARAMS.items() if k != "n_jobs"}

regularisation sweep, still inside the training clients only
                                   candidate                features  p50_mean  p50_worst_fold  auc_mean
          random forest, min_samples_leaf=20     5 contract features     0.904           0.780     0.707
          random forest, min_samples_leaf=20 + momentum-missing flag     0.920           0.760     0.709
         random forest, min_samples_leaf=100     5 contract features     0.912           0.800     0.712
         random forest, min_samples_leaf=100 + momentum-missing flag     0.920           0.840     0.711
         random forest, min_samples_leaf=400     5 contract features     0.920           0.880     0.710
         random forest, min_samples_leaf=400 + momentum-missing flag     0.920           0.880     0.708
        random forest, min_samples_leaf=1600     5 contract features     0.924           0.880     0.700
        random forest, min_samples_leaf=1600 + momentum-missing flag     0.924           0.860     

**Reading the plateau, not the peak.**

Every forest setting in the sweep lands between **0.904 and 0.924** mean precision@50. The whole
range is *one page out of fifty*. Picking the exact argmax of a sweep that noisy would be picking
noise and calling it tuning, so I take a setting in the middle of the flat region —
**`min_samples_leaf=400`** (mean 0.920) — rather than the nominal winner at leaf=1600 (0.924).
The tie-breaker I do trust is the **worst fold**, because it is the one that stands in for "an
odd client arrives": leaf=400 holds up at **0.880** against 0.780 for the untuned forest, and it
is level with leaf=1600 there while sitting in the middle of the plateau rather than at its edge.

Three smaller findings from the same sweep:

- **More capacity does not help.** The un-regularised forest (leaf=20) is *worse* on both the mean
  (0.904) and the floor (0.780) than the heavily regularised one. With five features, one month
  pair and a concentrated panel, extra flexibility gets spent memorising clients — the overfitting
  the grouped split exists to expose.
- **Boosting got worse the more I tuned it.** Its three settings run 0.889 → 0.884 → 0.880 as the
  regularisation goes up, so its best configuration is the plain one it started with. That is the
  setting it enters the comparison table with — as a second opinion, not as my pick.
- **The sixth feature buys nothing at the chosen setting.** `has_first_half_traffic` — the
  missingness flag w03 built, inspected, and deliberately parked as *context* because the contract
  caps this lane at five features — leaves the score at exactly **0.920 → 0.920** at leaf=400, and
  moves it between 0.0 and +1.6pp elsewhere. That is inside the noise. **I keep the five-feature
  contract**: the cap was declared before I saw any score, and a rule you only keep when it is free
  is not a rule. Recorded here so the choice is auditable rather than silent.

The model I commit to — before the held-out clients are opened — is the random forest above.

## 2. Split design

**One split, inherited, not chosen this week.** `GroupShuffleSplit(test_size=0.3,
random_state=42)` grouped by `client_hash_id` — 18 training clients, 9 held-out clients, every
page of a client on exactly one side. It was fixed in w03, used to freeze the baseline in w04, and
re-derived in section 0c. Choosing a split *after* seeing a model score is how a comparison
quietly stops being one.

**Why grouped by client — and what that question actually is.** A split is a deployment
question: it should imitate the gap between what the model learned and what it will meet. Pages
inside one client are not independent draws — they share a site template, an editorial calendar, a
CMS migration, a redesign. A random *row* split would put pages from the same client on both
sides, so a model could score well by recognising the client rather than the pattern. The cell
below measures exactly what that easier split would have bought me.

**What grouped-by-client cannot answer.** It tests *a new client*, not *a new month*. My label is
one March→April transition, so nothing here tells me whether the pattern survives to May — anything
that moved search broadly in April 2026 is inside every number in this notebook. That is a
time-split question, it needs a second month pair, and it is w06's job (ML-09), not a
reinterpretation of this one.

**Why the training clients are split again, inside.** Choosing between four candidates on the
held-out clients would make those clients part of the training procedure, and the "held-out" score
would be the score of the best of four guesses. So selection ran on grouped 5-fold CV inside the
18 training clients (section 1), and the 9 held-out clients are opened exactly once, in section 3,
with one model already named.

In [6]:
comp = (frame.groupby(["split", "client_hash_id"])
             .agg(pages=("label_declining", "size"), base_rate=("label_declining", "mean"))
             .reset_index())
print("split composition (client ids withheld — they are pseudonyms and stay out of the output)")
for part in ("train", "test"):
    p = comp[comp.split == part].sort_values("pages", ascending=False)
    print(f"\n{part}: {p.pages.sum():,} pages / {len(p)} clients | "
          f"base rate {frame[frame.split == part].label_declining.mean():.2%}")
    print(f"  page counts per client: {', '.join(f'{n:,}' for n in p.pages)}")
    print(f"  largest client is {p.pages.max() / p.pages.sum():.1%} of this side")
print(f"\nclients appearing on both sides: "
      f"{len(set(train.client_hash_id) & set(test.client_hash_id))}  (0 = the split is honest)")

# --- what the easier split would have bought --------------------------------
# Both cuts are made INSIDE the training clients, so the held-out clients stay sealed: this
# compares two ways of cutting the same 59,682 rows, not two peeks at the answer.
from sklearn.model_selection import ShuffleSplit

cuts = []
for tag, a, b in [
    ("random rows (the easy one)", *next(ShuffleSplit(n_splits=1, test_size=0.3,
                                                      random_state=SEED).split(train))),
    ("grouped by client (mine)",   *next(GroupShuffleSplit(n_splits=1, test_size=0.3,
                                                           random_state=SEED).split(train, groups=train.client_hash_id))),
]:
    A, B = train.iloc[a], train.iloc[b]
    risk = make_chosen().fit(A[FEATURES], A.label_declining).predict_proba(B[FEATURES])[:, 1]
    cuts.append({"split": tag, "val_rows": len(B),
                 "precision_at_50": precision_at_k_fair(B.assign(risk=risk), "risk")["tie_fair"],
                 "auc": roc_auc_score(B.label_declining, risk),
                 "clients_on_both_sides": len(set(A.client_hash_id) & set(B.client_hash_id))})

cuts = pd.DataFrame(cuts)
print("\nthe same model, the same training rows, two ways of cutting them:")
print(cuts.round(3).to_string(index=False))
print(f"\nthe easier split is worth +{cuts.auc.iloc[0] - cuts.auc.iloc[1]:.3f} AUC and "
      f"+{(cuts.precision_at_50.iloc[0] - cuts.precision_at_50.iloc[1]) * 100:.0f}pp precision@50 — "
      f"bought entirely by letting the model meet {cuts.clients_on_both_sides.iloc[0]} of its own "
      f"training clients again on the other side.")

receipts["split_contrast"] = {r.split: {"precision_at_50": round(r.precision_at_50, 4),
                                        "auc": round(r.auc, 4)} for r in cuts.itertuples()}

split composition (client ids withheld — they are pseudonyms and stay out of the output)

train: 59,682 pages / 18 clients | base rate 55.69%
  page counts per client: 17,313, 12,764, 8,501, 7,996, 7,528, 1,844, 1,458, 958, 642, 478, 122, 35, 18, 11, 6, 4, 2, 2
  largest client is 29.0% of this side

test: 25,771 pages / 9 clients | base rate 49.36%
  page counts per client: 21,633, 1,646, 1,466, 758, 192, 42, 19, 14, 1
  largest client is 83.9% of this side

clients appearing on both sides: 0  (0 = the split is honest)

the same model, the same training rows, two ways of cutting them:
                     split  val_rows  precision_at_50   auc  clients_on_both_sides
random rows (the easy one)     17905              1.0 0.766                     17
  grouped by client (mine)     10365              0.8 0.611                      0

the easier split is worth +0.155 AUC and +20pp precision@50 — bought entirely by letting the model meet 17 of its own training clients again on the other sid

**Verdict: the grouped split is the honest one, and it costs me real points to keep it.**

The identical model, on the identical 59,682 training rows, scores **precision@50 = 100.0% and AUC
0.766** when the cut lets it meet 17 of its own clients again on the other side — and **80.0% /
0.611** when it has to face six clients it has never seen. Same model, same rows, **20 percentage
points** of difference, produced entirely by how the rows were divided. Neither number is wrong;
they answer different questions. "How well would this rank pages for a client I already serve?" is
the flattering one. "How well would this rank pages for a client that onboards next month?" is the
one an editor is actually asking, and it is the smaller number. I report the smaller one everywhere
below — and it is worth sitting with the fact that the easy split hands back a **perfect fifty out
of fifty**, which is exactly the kind of number that ends up in a slide deck.

The composition print is the caveat that travels with every score in section 3: the held-out side
is **9 clients, and one of them is 84% of its pages**. Precision@50 on this test set is, in
practice, mostly a measurement about a single client — which is why section 3 also reads the
score per client, and why section 3's stability check re-cuts the split 25 times instead of
trusting this one.

## 3. Train + compare vs my baseline

**The comparison contract.** Same 25,771 held-out rows, same 9 clients,
same metric (precision@50), same notebook run. The frozen rule is in the table, not in a footnote.
Alongside it sit two references that keep the score honest: **picking 50 pages at random** (what
an editor without a queue would get) and **"steepest slide only"**, the single strongest signal
from w04's audit — because a model that cannot beat one column does not need five.

Two columns in the table are not about precision at all, and they are there on purpose:

- **`imp_protected`** — the March→April impressions actually lost by the 50 pages each queue
  picked. Precision counts *pages*; an editor spends their day protecting *traffic*. w04 already
  found these two can point in opposite directions.
- **`tied_at_50`** — how many pages share the score sitting at rank 50. Where that number is
  large, the queue was not chosen by the model; it was chosen by row order. The `p@K` columns are
  therefore tie-fair (the expected precision over the tied block), and `as_listed_50` shows what
  `nlargest` would have printed instead.

In [7]:
MODELS = {
    "logistic regression":     make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
    "decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=SEED),
    CHOSEN_NAME:               make_chosen(),
    "gradient boosting":       make_boosting(),
}
for name, model in MODELS.items():
    model.fit(train[FEATURES], train.label_declining)                 # trained on training clients only
    test[name] = model.predict_proba(test[FEATURES])[:, 1]            # scored on the sealed clients

test["lost_impressions"] = (test.imp_mar - test.imp_apr).clip(lower=0)   # label-side, reporting only
test["steepest_slide"] = -test.momentum_in_month                         # the best single signal from w04

SCORERS = [("rule baseline (w04, frozen)", "action_score"),
           ("steepest slide only (1 column)", "steepest_slide")] + [(n, n) for n in MODELS]

rows = []
for label, col in SCORERS:
    q = test.nlargest(TOP_K, col)
    at_50 = precision_at_k_fair(test, col)
    row = {"scorer": label}
    row.update({f"p@{k}": precision_at_k_fair(test, col, k=k)["tie_fair"] for k in (10, 25, 50, 100, 250)})
    row.update({"as_listed_50": at_50["as_listed"], "tied_at_50": at_50["tied_at_cut"],
                "auc": roc_auc_score(test.label_declining, test[col]),
                "median_imp": int(q.imp_mar.median()),
                "imp_protected": int(q.lost_impressions.sum()),
                "clients": int(q.client_hash_id.nunique())})
    rows.append(row)

rng = np.random.default_rng(SEED)
draws = [test.iloc[rng.choice(len(test), TOP_K, replace=False)] for _ in range(200)]
rows.append({"scorer": "picking 50 at random (200 draws)",
             "p@50": float(np.mean([d.label_declining.mean() for d in draws])), "auc": 0.5,
             "median_imp": int(np.median([d.imp_mar.median() for d in draws])),
             "imp_protected": int(np.mean([d.lost_impressions.sum() for d in draws])),
             "clients": int(np.median([d.client_hash_id.nunique() for d in draws]))})

comparison = pd.DataFrame(rows)
print(f"HELD-OUT CLIENTS — {len(test):,} pages / {test.client_hash_id.nunique()} clients | "
      f"base rate {test.label_declining.mean():.2%}")
print("p@K columns are tie-fair; imp_protected = March-to-April impressions lost by those 50 pages\n")
print(comparison.round(3).to_string(index=False))

p95 = float(np.percentile([d.label_declining.mean() for d in draws], 95))
print(f"\nrandom picking: mean {np.mean([d.label_declining.mean() for d in draws]):.1%}, "
      f"95th percentile {p95:.1%} — the floor any queue has to clear to mean anything.")
print(f"model minus baseline at precision@{TOP_K}: "
      f"{(precision_at_k_fair(test, CHOSEN_NAME)['tie_fair'] - BASELINE_P50) * 100:+.1f}pp "
      f"({(precision_at_k_fair(test, CHOSEN_NAME)['tie_fair'] - BASELINE_P50) * TOP_K:+.0f} pages out of {TOP_K})")

receipts["comparison_test"] = {r["scorer"]: round(r.get("p@50", float("nan")), 4) for r in rows}
receipts["comparison_auc"] = {r["scorer"]: round(r.get("auc", float("nan")), 4) for r in rows}
receipts["comparison_imp_protected"] = {r["scorer"]: int(r["imp_protected"]) for r in rows}
receipts["random_top50_mean"] = round(float(np.mean([d.label_declining.mean() for d in draws])), 4)
receipts["random_top50_p95"] = round(p95, 4)

HELD-OUT CLIENTS — 25,771 pages / 9 clients | base rate 49.36%
p@K columns are tie-fair; imp_protected = March-to-April impressions lost by those 50 pages

                          scorer  p@10  p@25  p@50  p@100  p@250  as_listed_50  tied_at_50   auc  median_imp  imp_protected  clients
     rule baseline (w04, frozen) 0.800 0.880 0.900  0.910  0.912          0.90         1.0 0.627        5698         320709        4
  steepest slide only (1 column) 0.900 0.880 0.920  0.960  0.944          0.92         1.0 0.631         402         117462        5
             logistic regression 0.600 0.520 0.560  0.580  0.568          0.56         1.0 0.588         124           1764        2
         decision tree (depth 3) 0.718 0.718 0.718  0.718  0.718          0.80      4836.0 0.640         776          49825        5
          random forest (chosen) 0.800 0.920 0.900  0.930  0.916          0.90         1.0 0.653        1455          53233        2
               gradient boosting 0.900 0.960 0

**Reading the table — the model did not beat the baseline.**

**Precision@50: rule 90.0%, chosen model 90.0%.** Identical. Forty-five correct pages out of fifty
either way, against a 49.4% base rate and a random-draw 95th percentile of 60%. Both are far above
what an editor without a queue would get; neither is above the other. After a method menu, a
regularisation sweep and grouped cross-validation, **the headline of this week is a tie.**

Four things the table says that a single number would have hidden:

- **The model ranks better overall while ranking no better at the head.** AUC goes from 0.627
  (rule) to 0.653 (forest) to 0.666 (boosting) — a real improvement in ordering all 25,771 pages.
  It buys nothing at rank 50. That is not a paradox: AUC rewards getting the whole list roughly
  right, precision@50 only asks about the fifty pages anyone will actually open. My lane declared
  precision@50 in w01 and I am not switching metrics now that AUC flatters the model.
- **The rule protects 6x more traffic at the same precision.** Its 50 pages had a median of 5,698
  March impressions and bled **320,709** impressions; the forest's 50 had a median of 1,455 and
  bled **53,233**. Equal precision, very different days of work — the rule's explicit
  `log1p(imp_mar)` size factor is doing something the forest never learned to value, because
  precision@50 does not reward it.
- **One column still matches five.** "Steepest slide only" scores **92.0%** while protecting
  117,462 impressions from queue pages with a median size of 402. w04 found this and it survives:
  at the head of this queue the within-month slide *is* most of the signal.
- **The model I did not pick is the one that beat the rule.** Gradient boosting scores **92.0%**
  and protects **280,650** impressions — nominally the best row in the table on both counts, and
  ahead of the frozen rule by one page. I am not going to announce that as a win, for two reasons:
  it is not the model I committed to in section 1, and *one page* is exactly the size of difference
  the stability check below shows to be noise. It goes in the table because hiding it would be worse.

And the row that looks strangest is the one to read first: the depth-3 tree scores **71.8% at
every K from 10 to 250**, while `nlargest` printed **80%** for the same fifty slots.

In [8]:
print("=== tie audit: how many pages share the score sitting at rank 50? ===")
audit = pd.DataFrame([{"scorer": label, **precision_at_k_fair(test, col)} for label, col in SCORERS])
print(audit.round(3).to_string(index=False))

print("\n=== forensics on the depth-3 tree ===")
col = "decision tree (depth 3)"
cut = np.sort(test[col].to_numpy())[::-1][TOP_K - 1]
tied = test[test[col] == cut]
print(f"highest score the tree can give any page: {test[col].max():.4f}")
print(f"score at rank 50: {cut:.4f} -> the top 50 is not a ranking, it is the first 50 rows of a "
      f"{len(tied):,}-page block that all scored the same")
print(f"that block's own decline rate: {tied.label_declining.mean():.1%}")

shuffles = [tied.iloc[rng.choice(len(tied), TOP_K, replace=False)].label_declining.mean() for _ in range(500)]
print(f"drawing 50 from it at random, 500 times: mean {np.mean(shuffles):.1%}, "
      f"5th-95th percentile {np.percentile(shuffles, 5):.0%}-{np.percentile(shuffles, 95):.0%}, "
      f"best draw seen {max(shuffles):.0%}")
print(f"what pandas .nlargest actually returned: {precision_at_k(test, col):.0%}")

order = tied.groupby("client_hash_id").agg(pages=("label_declining", "size"),
                                           decline_rate=("label_declining", "mean"),
                                           first_row_position=("label_declining", lambda s: s.index.min()))
print("\nthe tied block, by client, in the order the parquet scan happens to return them "
      "(ids withheld):")
print(order.sort_values("first_row_position").reset_index(drop=True).round(3).to_string())
print(f"\nthe first 50 rows of that block come from {tied.head(TOP_K).client_hash_id.nunique()} client(s) "
      f"and decline at {tied.head(TOP_K).label_declining.mean():.0%}.")

receipts["tie_audit"] = {label: precision_at_k_fair(test, col)["tied_at_cut"] for label, col in SCORERS}
receipts["tree_tied_block"] = {"pages": int(len(tied)), "block_decline_rate": round(float(tied.label_declining.mean()), 4),
                               "as_listed": round(precision_at_k(test, col), 4),
                               "random_draw_mean": round(float(np.mean(shuffles)), 4)}

=== tie audit: how many pages share the score sitting at rank 50? ===
                        scorer  as_listed  tie_fair  tied_at_cut  slots_from_ties
   rule baseline (w04, frozen)       0.90     0.900            1                1
steepest slide only (1 column)       0.92     0.920            1                1
           logistic regression       0.56     0.560            1                1
       decision tree (depth 3)       0.80     0.718         4836               50
        random forest (chosen)       0.90     0.900            1                1
             gradient boosting       0.92     0.920            1                1

=== forensics on the depth-3 tree ===
highest score the tree can give any page: 0.8482
score at rank 50: 0.8482 -> the top 50 is not a ranking, it is the first 50 rows of a 4,836-page block that all scored the same
that block's own decline rate: 71.8%
drawing 50 from it at random, 500 times: mean 71.4%, 5th-95th percentile 60%-82%, best draw seen 88%
wh

**Exact ties chose that queue, not the model.**

A depth-3 tree has 8 leaves, so it can emit **8 distinct scores** for 25,771 pages. Its best leaf
holds **4,836** held-out pages that all score 0.8482 — so the top 50, the top 100 and the top 250
are all the *same heap*, which is why its tie-fair precision is **71.8% at every K in the table**.
It is not ranking; it is sorting into piles, and a queue needs a ranking.

Which 50 of those 4,836 pages an editor sees was decided by `nlargest`, which breaks ties by row
order. Row order is parquet scan order, and scan order groups pages by client — clients whose
decline rates inside this one block run from **25.0% to 96.7%**. Drawing 50 from the block at
random 500 times gives a mean of 71.4%, a 5th-95th percentile of **60% to 82%**, and a best draw of
88%. So the printed number for this model could honestly have been anywhere in that range: it
happened to be **80%**, and it is measuring the order of rows in a file.

This is why the tie audit sits in the comparison table and not in an appendix. My own rule survives
the same check for a boring reason — exactly one page ties at its rank-50 cut, because a continuous
score with a `log1p(imp_mar)` term almost never repeats. But the rule has **16,556 held-out pages
tied at exactly zero**, so if editorial capacity were 20,000 rather than 50 its queue would be
chosen by scan order too. Any score with flat regions needs a stated tie-break (biggest page first,
say) before it reaches a human — and, as section 0b notes, it needs a stable row order underneath
it, which is a thing you have to ask a parallel query engine for explicitly.

In [9]:
from sklearn.model_selection import GroupShuffleSplit

runs = []
for seed in range(25):
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
                .split(frame, groups=frame.client_hash_id))
    A, B = frame.iloc[a].copy(), frame.iloc[b].copy()
    med = band_median_ctr(A)                                  # the rule refits its one quantity
    B["rule"] = score_pages(B, med)
    B["model"] = make_chosen().fit(A[FEATURES], A.label_declining).predict_proba(B[FEATURES])[:, 1]
    B["boosting"] = make_boosting().fit(A[FEATURES], A.label_declining).predict_proba(B[FEATURES])[:, 1]
    runs.append({"seed": seed, "held_out_clients": B.client_hash_id.nunique(),
                 "base_rate": B.label_declining.mean(),
                 **{k: precision_at_k_fair(B, k)["tie_fair"] for k in ("rule", "model", "boosting")}})

runs = pd.DataFrame(runs)
runs["model_minus_rule"] = runs.model - runs.rule
runs["boosting_minus_rule"] = runs.boosting - runs.rule
print("25 fresh grouped splits — the rule and both models are rebuilt from scratch on each one")
print(runs.round(3).to_string(index=False))
for who in ("rule", "model", "boosting"):
    print(f"{who:>9}: median {runs[who].median():.3f}  worst {runs[who].min():.3f}  best {runs[who].max():.3f}")

for who in ("model", "boosting"):
    d = runs[f"{who}_minus_rule"]
    print(f"\n{who} vs rule: wins {int((d > 0).sum())}/25, ties {int((d == 0).sum())}, loses {int((d < 0).sum())}"
          f" | median {d.median():+.3f}, mean {d.mean():+.3f}, "
          f"5th-95th percentile {np.percentile(d, 5):+.3f} to {np.percentile(d, 95):+.3f}")

receipts["stability_25_splits"] = {
    who: {"median": round(float(runs[who].median()), 4), "min": round(float(runs[who].min()), 4),
          "max": round(float(runs[who].max()), 4)} for who in ("rule", "model", "boosting")}
receipts["stability_vs_rule"] = {
    who: {"wins": int((runs[f"{who}_minus_rule"] > 0).sum()),
          "ties": int((runs[f"{who}_minus_rule"] == 0).sum()),
          "losses": int((runs[f"{who}_minus_rule"] < 0).sum()),
          "median_difference": round(float(runs[f"{who}_minus_rule"].median()), 4)}
    for who in ("model", "boosting")}

25 fresh grouped splits — the rule and both models are rebuilt from scratch on each one
 seed  held_out_clients  base_rate  rule  model  boosting  model_minus_rule  boosting_minus_rule
    0                 9      0.428  0.92   0.96     0.920              0.04                0.000
    1                 9      0.697  0.98   0.98     0.980              0.00                0.000
    2                 9      0.546  0.94   0.92     0.900             -0.02               -0.040
    3                 9      0.481  0.94   0.90     0.940             -0.04                0.000
    4                 9      0.543  0.94   1.00     0.940              0.06                0.000
    5                 9      0.623  0.96   0.94     0.980             -0.02                0.020
    6                 9      0.438  0.96   0.96     0.840              0.00               -0.120
    7                 9      0.442  0.96   0.90     0.840             -0.06               -0.120
    8                 9      0.533  0.7

**Is the tie a fluke of one split? No — the tie is the stable answer. And boosting's win is not.**

Re-cutting the same 27 clients 25 different ways and rebuilding all three scorers each time:

- **Forest vs rule: 8 wins, 6 ties, 11 losses.** Median difference **exactly 0.000**, 5th-95th
  percentile **-5.6pp to +9.6pp**. There is no version of this comparison in which the forest is
  reliably ahead — the frozen split's tie is the representative outcome, not a coincidence.
- **Boosting vs rule: 5 wins, 4 ties, 16 losses**, median **-4.0pp**. The model that looked best on
  the sealed split is the one that loses to the rule most often across re-cuts, and its median
  score (0.900) is the lowest of the three. Its +2pp on the frozen split was a lucky draw, not a
  discovery — which is precisely why I did not promote it two cells ago, and why the promotion
  would have been so tempting without this table.

The second thing this table shows is how thin a 50-page measurement is. The frozen rule's
precision@50 ranges from **70% to 98%** depending purely on which nine clients happen to be held
out. That range is not the rule getting better or worse — it is the same rule meeting different
clients. So "90.0% vs 90.0%" in the comparison table should be read as *these two scorers are
indistinguishable at this sample size*, not as a dead heat measured to the decimal. Any claim that
one beat the other by two or three points would be a claim about which clients I drew.

**One split is one draw.** That sentence belongs next to every precision@K in this lane.

In [10]:
per_client_k = 5
print(f"the same scorers, but as a per-client queue: top {per_client_k} pages for each held-out client")
rows = []
for label, col in SCORERS:
    q = test.sort_values(col, ascending=False).groupby("client_hash_id").head(per_client_k)
    g = test.nlargest(TOP_K, col)
    rows.append({"scorer": label, "per_client_precision": q.label_declining.mean(),
                 "per_client_slots": len(q),
                 "per_client_imp_protected": int(q.lost_impressions.sum()),
                 "global_precision_at_50": precision_at_k_fair(test, col)["tie_fair"],
                 "clients_served_by_global_queue": int(g.client_hash_id.nunique())})
print(pd.DataFrame(rows).round(3).to_string(index=False))

pc = (test.groupby("client_hash_id")
          .agg(pages=("label_declining", "size"), base_rate=("label_declining", "mean"))
          .assign(in_rule_top50=test.nlargest(TOP_K, "action_score").groupby("client_hash_id").size(),
                  in_model_top50=test.nlargest(TOP_K, CHOSEN_NAME).groupby("client_hash_id").size())
          .fillna(0).astype({"in_rule_top50": int, "in_model_top50": int})
          .sort_values("pages", ascending=False))
print("\nwhere the 50 slots actually go, per held-out client (ids withheld):")
print(pc.reset_index(drop=True).round(3).to_string())

receipts["per_client_queue"] = {r["scorer"]: round(float(r["per_client_precision"]), 4) for r in rows}
receipts["largest_client_share_of_test"] = round(float(pc.pages.max() / pc.pages.sum()), 4)

the same scorers, but as a per-client queue: top 5 pages for each held-out client
                        scorer  per_client_precision  per_client_slots  per_client_imp_protected  global_precision_at_50  clients_served_by_global_queue
   rule baseline (w04, frozen)                 0.707                41                     71667                   0.900                               4
steepest slide only (1 column)                 0.732                41                     98557                   0.920                               5
           logistic regression                 0.537                41                      2475                   0.560                               2
       decision tree (depth 3)                 0.659                41                     14509                   0.718                               5
        random forest (chosen)                 0.683                41                     18637                   0.900                               2


**A global queue and a per-client queue are different jobs, and my metric only grades one.**

Forced to give every held-out client its own five pages, precision drops for everything: the rule
falls from 90.0% to **70.7%**, the forest from 90.0% to **68.3%**, boosting from 92.0% to 78.0%.
The same scores, the same pages, a different question — and 14 to 22 points harder to be right
about.

The per-client table shows why. **One client holds 84% of the held-out pages, and takes 45 of the
rule's 50 slots and 48 of the forest's 50.** A global top-50 in this panel is, in effect, a queue
for the biggest account. Meanwhile a client with 1,646 pages and a **90.9% decline rate** receives
**zero** slots from either scorer — the pages that are bleeding there are simply smaller or flatter
than the winner's, so a global ranking never reaches them.

If FlyRank ships one list per account manager, the number that matters is the 70.7% column, not
the 90.0% one. My lane declared a global precision@50 in w01 and I am reporting it, but this table
is the honest footnote: **the metric I chose grades the queue I am not sure the product would
ship.** Deciding which of the two the editor actually gets is a product question.

## 4. Errors and interpretation

A metric without error analysis is decoration, and this week's metric is a tie — so the
interesting question is no longer "which number is bigger" but **"what did the model learn, where
is it wrong, and is there any job it does that my rule cannot?"** Four passes, in that order:
what it leans on, where it breaks, what its wrong picks look like up close, and the one place it
genuinely adds something.

The first check is the leakage sanity check: *does the top feature
make sense, or is it suspiciously perfect?* Importance is measured by **shuffling** each column on
the held-out clients and watching the score fall — measured twice, once on AUC and once on the
metric I actually ship.

In [11]:
from sklearn.inspection import permutation_importance
from sklearn.tree import export_text

rf = MODELS[CHOSEN_NAME]


def p50_scorer(estimator, X, y):
    """permutation_importance, judged on the metric the lane actually ships."""
    proba = estimator.predict_proba(X)[:, 1]
    scored = pd.DataFrame({"label_declining": np.asarray(y), "risk": proba})
    return precision_at_k_fair(scored, "risk")["tie_fair"]


imp = {}
for tag, scoring in [("AUC", "roc_auc"), (f"precision@{TOP_K}", p50_scorer)]:
    r = permutation_importance(rf, test[FEATURES], test.label_declining, scoring=scoring,
                               n_repeats=10, random_state=SEED, n_jobs=-1)
    imp[tag] = pd.Series(r.importances_mean, index=FEATURES)
    imp[tag + " sd"] = pd.Series(r.importances_std, index=FEATURES)

print("permutation importance on the HELD-OUT clients — how far the score falls when one column "
      "is shuffled")
print(pd.DataFrame(imp).sort_values("AUC", ascending=False).round(4).to_string())
print("\nthe forest's own (train-fitted) impurity importances, for contrast:")
print(pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False).round(4).to_string())

print("\n=== the depth-3 tree, printed in full — the model you can read ===")
print(export_text(MODELS["decision tree (depth 3)"], feature_names=FEATURES, decimals=3, show_weights=True))

print("=== logistic regression coefficients (standardised units) ===")
coefs = pd.Series(MODELS["logistic regression"][-1].coef_[0], index=FEATURES)
print(coefs.sort_values(key=abs, ascending=False).round(3).to_string())
print("\nand the spread of the same columns on the training clients:")
print(train[FEATURES].describe(percentiles=[.01, .5, .99]).T[["min", "1%", "50%", "99%", "max"]].round(3).to_string())

receipts["permutation_importance_auc"] = {k: round(float(v), 4) for k, v in imp["AUC"].items()}
receipts["permutation_importance_p50"] = {k: round(float(v), 4) for k, v in imp[f"precision@{TOP_K}"].items()}

permutation importance on the HELD-OUT clients — how far the score falls when one column is shuffled
                      AUC  AUC sd  precision@50  precision@50 sd
momentum_in_month  0.1055  0.0024         0.308           0.0711
ctr_mar            0.0381  0.0018        -0.020           0.0310
pos_mar            0.0097  0.0006        -0.016           0.0250
log_imp_mar        0.0085  0.0003        -0.016           0.0388
active_days        0.0025  0.0013        -0.028           0.0183

the forest's own (train-fitted) impurity importances, for contrast:
momentum_in_month    0.5521
ctr_mar              0.1691
active_days          0.1566
pos_mar              0.0699
log_imp_mar          0.0524

=== the depth-3 tree, printed in full — the model you can read ===
|--- momentum_in_month <= 0.975
|   |--- momentum_in_month <= 0.674
|   |   |--- ctr_mar <= 0.003
|   |   |   |--- weights: [1539.000, 8597.000] class: 1
|   |   |--- ctr_mar >  0.003
|   |   |   |--- weights: [778.000, 2017.000] cl

**What it leans on: the same column my rule leans on, and almost nothing else.**

Shuffling `momentum_in_month` costs the forest **0.105 AUC** and **30.8 points of precision@50**.
`ctr_mar` is a distant second on AUC (0.038) and the rest are near zero. At the head of the queue
the picture is starker still: **every other column has a negative importance** — shuffling
`active_days` *improves* precision@50 by 2.8 points, `ctr_mar` by 2.0, `pos_mar` and `log_imp_mar`
by 1.6 each. Negative importance is the honest way of saying "this column is noise here, and
randomising it happened to help"; all four sit within about one standard deviation of zero. **The
top-50 queue rests on one column.**

Three things follow.

1. **No leakage smell.** The top feature is strong but not suspiciously perfect: a page whose
   second half of March ran far below its first half is more likely to post a lower April total.
   That is a real, decision-time-knowable pattern — and w04 already named its limit, which is that
   it partly measures *persistence of a slide already under way* rather than the onset of a new
   one. An AUC of 0.653 is exactly what a genuine-but-partial signal looks like; a leaked column
   would have printed 0.99, as w03 demonstrated on purpose.
2. **The model rediscovered my rule.** Both scorers rank on the same axis, which is why their
   precision is identical and why nothing about the comparison is surprising in hindsight. Where
   they differ is that the rule *also* multiplies by page size on purpose, and the forest has
   learned that size is not predictive of a 20% drop — which is true, and is also why its queue
   protects a sixth of the traffic.
3. **The impurity importances disagree, and they are the ones to distrust.** The train-fitted
   ranking spreads credit across `active_days` (0.157) and `pos_mar` (0.070) — columns that
   contribute nothing, or slightly less than nothing, on held-out clients. Impurity importance
   rewards columns with many split points; permutation importance on unseen data asks whether the
   column pays. Reporting the first one would have made this model look like it uses five features.

**Why logistic regression failed:** its coefficient on `momentum_in_month` is
**-0.036**, the smallest in the model — on the strongest signal in the data. The column runs from
0.001 to 4,091 with a median near 1.0, so standardising it crushes the entire informative range
(collapsed → flat) into a sliver of the axis, and a straight line in that space cannot express
"0.1 is far worse than 0.9". Trees split on *order*, not distance, so the identical column works
for them untouched. That is a lesson about the feature, not about linear models: had I log-scaled
momentum in w03, the linear model would have had a fair fight. I am not changing the contract now
to give it one.

**And the printed tree is the most useful model in this notebook even though it scored worst.**
Its first split is `momentum_in_month <= 0.975`, its second is `ctr_mar` — the same two signals,
in the same order, that w04's audit picked by hand, discovered independently from the label. A
tree that reproduces your rule's first two signals, in your rule's order, is evidence the rule was
reading the data and not the analyst.

In [12]:
test["mom_bucket"] = pd.cut(test.momentum_in_month, [-0.01, 0.5, 0.8, 1.2, 2.0, 10**9],
                            labels=["<0.5", "0.5-0.8", "0.8-1.2", "1.2-2.0", ">2.0"])
test["size_bucket"] = pd.qcut(test.imp_mar, [0, .25, .5, .75, .9, 1.],
                              labels=["p0-25", "p25-50", "p50-75", "p75-90", "p90-100"])
test["pos_band"] = pd.cut(test.pos_mar, POS_BANDS, labels=POS_NAMES)
test["predicted"] = (test[CHOSEN_NAME] >= 0.5).astype(int)


def error_slice(col):
    g = test.groupby(col, observed=True)
    out = g.agg(pages=("label_declining", "size"), actual_decline=("label_declining", "mean"),
                mean_predicted=(CHOSEN_NAME, "mean"))
    out["false_alarm_rate"] = g.apply(
        lambda d: float(((d.predicted == 1) & (d.label_declining == 0)).sum() / max((d.predicted == 1).sum(), 1)),
        include_groups=False)
    out["missed_declines"] = g.apply(
        lambda d: float(((d.predicted == 0) & (d.label_declining == 1)).sum() / max(d.label_declining.sum(), 1)),
        include_groups=False)
    return out


for col in ("mom_bucket", "size_bucket", "pos_band"):
    print(f"--- errors by {col} (chosen model, held-out clients, threshold 0.5) ---")
    print(error_slice(col).round(3).to_string())
    print()

print("--- is the probability a probability? (it is used as a rank, but worth knowing) ---")
test["prob_bin"] = pd.cut(test[CHOSEN_NAME], [0, .2, .4, .5, .6, .7, .8, 1.0])
print(test.groupby("prob_bin", observed=True)
          .agg(pages=("label_declining", "size"), mean_predicted=(CHOSEN_NAME, "mean"),
               actually_declined=("label_declining", "mean")).round(3).to_string())

receipts["missed_declines_by_momentum"] = {str(k): round(float(v), 4)
                                           for k, v in error_slice("mom_bucket").missed_declines.items()}

--- errors by mom_bucket (chosen model, held-out clients, threshold 0.5) ---
            pages  actual_decline  mean_predicted  false_alarm_rate  missed_declines
mom_bucket                                                                          
<0.5         2878           0.796           0.847             0.204            0.000
0.5-0.8      6352           0.582           0.756             0.418            0.001
0.8-1.2      9878           0.409           0.587             0.552            0.185
1.2-2.0      4849           0.378           0.492             0.573            0.424
>2.0         1814           0.479           0.418             0.417            0.737

--- errors by size_bucket (chosen model, held-out clients, threshold 0.5) ---
             pages  actual_decline  mean_predicted  false_alarm_rate  missed_declines
size_bucket                                                                          
p0-25         6450           0.533           0.602             0.441         

**Where it is wrong: the model inherits my rule's blind spot almost exactly.**

Sorted by the within-month slide, the miss rate is a staircase: of the pages that actually
declined, the model misses **0%** of those that had already halved inside March, **18.5%** of the
flat ones, **42.4%** of those that grew mildly, and **73.7%** of those that more than doubled. Its
false-alarm rate runs the other way — **55.2%** of its alarms on flat pages are wrong, against
20.4% on collapsing ones. The model, like the rule, is fluent about pages in motion and nearly
mute about pages that look fine and then drop. Learning five features did not fix that, because
the answer is not in those five features.

By page size the pattern is a warning for the product: the **largest decile draws the most false
alarms (52.5%)** — the pages where an editor's wasted afternoon costs the most are the ones the
model is least sure about. Position is the flattest slice of the three: false alarms run 33% at
positions 1-3 and 51% past position 20, and miss rates barely move at all, which is a polite way
of saying position is not carrying much here.

**The probabilities are not calibrated and should not be shown to anyone.** Pages the model scores
around 0.75 decline **53.6%** of the time; the 0.8+ band declines **73.7%**. It is systematically
overconfident, which is normal for a forest on a panel like this and harmless for a queue
(ranking only needs the order to be right). It stops being harmless the moment someone writes
"85% likely to decline" next to a page in a client-facing report. If this ships, it ships as
positions, not percentages — or it gets calibrated first, on a fold reserved for it.

In [13]:
def evidence(df, score_col):
    """Only decision-time columns, plus what April did — printed after the judgement, not before."""
    return (df.assign(april_vs_march=(df.imp_apr / df.imp_mar).round(2),
                      ctr_pct=(df.ctr_mar * 100).round(3), score=df[score_col].round(3))
              [["score", "imp_mar", "momentum_in_month", "pos_mar", "ctr_pct", "active_days",
                "spike_share", "april_vs_march"]].round(3))


top = test.nlargest(TOP_K, CHOSEN_NAME)
misses = top[top.label_declining == 0]
print(f"=== the model's wrong picks: {len(misses)} of its top {TOP_K} did not decline ===")
print(evidence(misses, CHOSEN_NAME).to_string(index=False))

rule_top = test.nlargest(TOP_K, "action_score")
rule_misses = rule_top[rule_top.label_declining == 0]
print(f"\n=== the rule's wrong picks, for contrast: {len(rule_misses)} of {TOP_K} ===")
print(evidence(rule_misses, "action_score").to_string(index=False))

print("\n=== the five biggest bleeds in the held-out set, and where each scorer ranked them ===")
big = test.nlargest(5, "lost_impressions").copy()
big["rule_rank"] = [int((test.action_score > v).sum() + 1) for v in big.action_score]
big["model_rank"] = [int((test[CHOSEN_NAME] > v).sum() + 1) for v in big[CHOSEN_NAME]]
print(big[["imp_mar", "imp_apr", "lost_impressions", "momentum_in_month", "rule_rank", "model_rank"]]
      .round(3).to_string(index=False))

receipts["model_top50_misses"] = int(len(misses))
receipts["rule_top50_misses"] = int(len(rule_misses))

=== the model's wrong picks: 5 of its top 50 did not decline ===
 score  imp_mar  momentum_in_month  pos_mar  ctr_pct  active_days  spike_share  april_vs_march
 0.945   1719.0              0.124    4.293    0.000           31        0.210            2.05
 0.943   2036.0              0.186    4.010    0.049           31        0.140            1.19
 0.940   1531.0              0.332    5.016    0.065           31        0.070            0.86
 0.939   2535.0              0.327    5.075    0.039           31        0.093            1.18
 0.939    942.0              0.132    5.243    0.000           31        0.795            1.25

=== the rule's wrong picks, for contrast: 5 of 50 ===
 score  imp_mar  momentum_in_month  pos_mar  ctr_pct  active_days  spike_share  april_vs_march
26.219  57720.0              0.162    6.019    0.074           31        0.093            1.03
25.573  50007.0              0.170    8.992    0.074           31        0.115            0.90
22.616   6143.0          

**Three wrong picks, up close, and why each one is hard.**

1. **The page that doubled instead of dying.** 1,719 March impressions, second half at 0.12x the
   first, position 4.3, zero clicks all month — every decision-time signal says collapse. April
   came in at **2.05x March**. Nothing available on 2026-03-31 distinguishes a page mid-collapse
   from a page mid-recovery; the shape of the evidence is identical.
2. **The spike leaving the window.** 942 impressions, 79.5% of them on a single March day. The
   "slide" is arithmetic: once the spike day rolls out of the second half, the ratio collapses on
   its own. w04 predicted this failure mode by hand, printed a spike warning next to the top-10,
   and deliberately left the guard out of the frozen rule. The model, which never saw
   `spike_share`, walks straight into it.
3. **The rule's expensive miss, which the model avoided.** A page with 6,143 March impressions and
   a 0.11 momentum that went to **267,960 impressions in April — a 43x jump**. The rule ranked it
   near the top of the queue; the forest did not. One page like this is worth more than the
   two-page precision differences the whole comparison table argues about, and it is the one
   concrete case where the model's flatter view of size did something useful.

All five of the model's misses share a profile — **small pages (942-2,535 impressions), steep
in-month slides, near-zero CTR, live all 31 days** — while the rule's misses are **large pages that
simply held steady** (57,720 and 50,007 impressions, April within 3% and 10% of March). Same
precision, opposite failure modes: the model wastes editor time on small pages that bounce, the
rule wastes it on big pages that were fine. Which error is cheaper is an editorial policy question,
not a modelling one, and it is worth putting to the team in exactly those terms.

One of the five is also a reminder that the label is a threshold, not a fact about the world: that
page came in at **0.86x March**, so it *did* lose a seventh of its traffic and still counts as a
miss, because the contract's line is a 20% drop. w03 named this when it defined the label; here is
what it costs in practice.

**And the biggest bleed of the month was found by the rule, not the model.** The page that fell
from 83,834 impressions to 188 sat at **rank 14** in the rule's queue and **rank 195** in the
model's — inside an editor's first fifty for one scorer, four queues deep for the other. The other
four biggest bleeds were invisible to both: they *grew* inside March before dropping, so the rule
scored three of them exactly zero (rank 9,216 — the top of its tie block) and the model buried them
between ranks 4,514 and 19,456.

In [14]:
blind = test[test.action_score == 0].copy()
print(f"=== the rule's blind spot: {len(blind):,} held-out pages it scores exactly 0 (not sliding in March) ===")
print(f"declines hiding in there: {int(blind.label_declining.sum()):,} — "
      f"{blind.label_declining.sum() / test.label_declining.sum():.1%} of every decline in the held-out set")
print(f"base rate inside the blind spot: {blind.label_declining.mean():.1%}\n")

rows = []
for label, col in [(n, n) for n in MODELS] + [("biggest pages first", "log_imp_mar")]:
    at_50 = precision_at_k_fair(blind, col)
    q = blind.nlargest(TOP_K, col)
    rows.append({"scorer": label, "precision_at_50": at_50["tie_fair"], "tied_at_50": at_50["tied_at_cut"],
                 "median_imp": int(q.imp_mar.median()), "imp_protected": int(q.lost_impressions.sum())})
rand = [blind.iloc[rng.choice(len(blind), TOP_K, replace=False)].label_declining.mean() for _ in range(200)]
rows.append({"scorer": "picking 50 at random in there", "precision_at_50": float(np.mean(rand))})
print("a SECOND queue, built only from pages the rule refuses to score:")
print(pd.DataFrame(rows).round(3).to_string(index=False))

print("\ndoes this hold on training clients, or is it one lucky test set? "
      "(same check, grouped 5-fold CV inside the 18 training clients)")
rows = []
for name in (CHOSEN_NAME, "gradient boosting"):
    folds = []
    for a, b in gkf.split(train, train.label_declining, groups=train.client_hash_id):
        A, B = train.iloc[a], train.iloc[b]
        model = make_chosen() if name == CHOSEN_NAME else make_boosting()
        model.fit(A[FEATURES], A.label_declining)
        Bb = B[B.action_score == 0].copy()
        Bb["risk"] = model.predict_proba(Bb[FEATURES])[:, 1]
        folds.append((precision_at_k_fair(Bb, "risk")["tie_fair"], Bb.label_declining.mean()))
    folds = np.array(folds)
    rows.append({"scorer": name, "blind_spot_p50_mean": folds[:, 0].mean(),
                 "per_fold": [round(v, 2) for v in folds[:, 0]], "blind_spot_base_rate": folds[:, 1].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

receipts["blind_spot"] = {
    "pages": int(len(blind)), "declines": int(blind.label_declining.sum()),
    "share_of_all_declines": round(float(blind.label_declining.sum() / test.label_declining.sum()), 4),
    "base_rate": round(float(blind.label_declining.mean()), 4),
    "chosen_model_precision_at_50": round(precision_at_k_fair(blind, CHOSEN_NAME)["tie_fair"], 4),
    "gradient_boosting_precision_at_50": round(precision_at_k_fair(blind, "gradient boosting")["tie_fair"], 4)}

=== the rule's blind spot: 16,556 held-out pages it scores exactly 0 (not sliding in March) ===
declines hiding in there: 6,746 — 53.0% of every decline in the held-out set
base rate inside the blind spot: 40.7%

a SECOND queue, built only from pages the rule refuses to score:
                       scorer  precision_at_50  tied_at_50  median_imp  imp_protected
          logistic regression            0.560         1.0       134.0         1846.0
      decision tree (depth 3)            0.484      3458.0      1189.0        46231.0
       random forest (chosen)            0.700         1.0       592.0        12414.0
            gradient boosting            0.600         1.0       822.0       137733.0
          biggest pages first            0.140         1.0     87275.0       372128.0
picking 50 at random in there            0.405         NaN         NaN            NaN

does this hold on training clients, or is it one lucky test set? (same check, grouped 5-fold CV inside the 18 training 

**The one job the model does that my rule cannot do at all.**

w04 ended with a measured gap: **53.0% of held-out declines happen on pages that were not sliding
inside March**, and the rule scores every one of them zero. Sixteen thousand pages, six thousand
seven hundred real declines, and a queue that will never show a single one of them.

Ranked *inside* that blind spot, the chosen forest reaches **70.0% precision@50** against a **40.7%**
base rate there — 35 correct picks in fifty where picking at random inside the same pool gets 20.
And unlike every other margin in this notebook, **this one reproduces**: the same forest averages
**71.2%** in the blind spot across grouped CV on the *training* clients, where the base rate is
44.9%. Two sides of the split, the same answer. So the model can see something real in pages the
rule declares uninteresting — a slower signal built from CTR, coverage and size rather than the
slide.

Two honest limits on it, though:

- **The level is soft even though the effect is not.** The per-fold values run 0.52 to 0.96. A
  +25 to +30 point lift over the base rate is what I would claim; the specific 70% is not.
- **Boosting is not better here, whatever the frozen split says.** It scores 60.0% inside the blind
  spot on the held-out clients and 65.2% in training-client CV — under the forest on both sides.
  Between this and the stability table, the case for promoting boosting on the strength of its
  92.0% headline has now failed three separate checks. That is what a pre-declared choice buys you:
  the temptation is visible instead of invisible.

Worth noting what the model does *not* fix in there either: **"biggest pages first" scores 14.0%
precision inside the blind spot while protecting 372,128 impressions** — 30x more traffic than the
forest's 12,414. The precision-versus-traffic tension from section 3 is sharper here, not softer.

That is still the most useful finding of the week, because it changes the shape of the product
rather than the size of a number: **the rule and the model are not competitors, they are two
queues.** The rule ranks pages already in motion, protects the most traffic, and is readable
enough for an editor to argue with. A model earns its keep on the pages the rule is silent about.
Testing that pairing properly — is the second queue worth an editor's time, on more than one month
pair — is w06 and w07's job.

---

### Conclusions worth knowing

**The model did not beat the baseline. Precision@50 is 90.0% for both, on the same 9 held-out
clients, in the same notebook run.** Across 25 re-cuts of the split, the forest wins 8, ties 6 and
loses 11, with a median difference of exactly zero. The rule protects **6x more traffic** per 50
picks (320,709 versus 53,233 impressions), and it is readable. **On this evidence I would ship the
rule and keep the model in the lab** — with one exception, below.

That is not a wasted week. Five things are now known that were not known previously:

1. **The learned model rediscovered the hand-written rule** — same top signal, same second signal,
   same order — which is the strongest evidence yet that w04's rule was reading the data rather
   than the analyst's taste.
2. **Complexity is not the bottleneck; the feature set is.** Regularising harder made the forest
   *better*, tuning made boosting *worse*, and both are mute on exactly the pages the rule is mute
   on. What would move the needle is a signal that speaks before the slide starts — query-level
   demand shift, a dated update history (w04 asked for this and the release cannot supply it), or
   several months of history per page — not a bigger model on these five columns.
3. **The model earns its keep in one specific place: the rule's blind spot.** 70.0% precision@50
   against a 40.7% base rate on held-out clients, 71.2% against 44.9% on training clients — the
   only margin in this notebook that reproduced on both sides of the split. That argues for **two
   queues, not a replacement**: the rule for pages already in motion, a model for the 53% of
   declines that start without a within-month warning.
4. **Three numbers in this notebook were nearly artefacts.** A depth-3 tree printed 80% where its
   real value is 71.8% and its plausible range was 60-88%, because 4,836 tied pages let file order
   choose the queue. The same forest scores a perfect 100% instead of 80% if the split is allowed
   to be easy. And boosting's 92% headline came from the scorer that loses to the rule in 16 of 25
   re-cuts. Each one would have read as a result in a slide deck.
5. **The metric may grade the wrong queue.** Per-client top-5 precision is 14 to 22 points below
   global precision@50 for every scorer, and one client owns 84% of the held-out pages.

**The smallest claim this evidence carries:** *on one month pair (March→April 2026), across 27
pseudonymised clients with nine held out, a random forest trained on five decision-time search
features ranked declining pages no better at the head of the queue than a transparent hand-written
rule (precision@50 = 90.0% for both, against a 49.4% base rate), and both ranked far better than
chance. On the subset the rule cannot score at all, the same model ranked measurably better than
chance on both sides of the split. Observed, directional, decision-support, one month pair — not a
forecast, not a causal statement, and not yet a claim about any other month.*

In [15]:
out_dir = pathlib.Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "w05_model_receipts.json"
receipts["verdict"] = "tie — model does not beat the frozen rule at precision@50 on held-out clients"
receipts["frozen_baseline_source"] = "work/outputs/w04_baseline_score_receipts.json"
out_path.write_text(json.dumps(receipts, indent=2), encoding="utf-8")
print(f"receipts written to work/outputs/{out_path.name} ({len(receipts)} entries)\n")
print(json.dumps({k: receipts[k] for k in [
    "chosen_model", "chosen_params", "baseline_precision_at_50_test", "test_base_rate",
    "stability_25_splits", "stability_vs_rule", "blind_spot", "verdict"]}, indent=2))

receipts written to work/outputs/w05_model_receipts.json (38 entries)

{
  "chosen_model": "RandomForestClassifier",
  "chosen_params": {
    "n_estimators": 300,
    "min_samples_leaf": 400,
    "random_state": 42
  },
  "baseline_precision_at_50_test": 0.9,
  "test_base_rate": 0.4936,
  "stability_25_splits": {
    "rule": {
      "median": 0.94,
      "min": 0.7,
      "max": 0.98
    },
    "model": {
      "median": 0.94,
      "min": 0.78,
      "max": 1.0
    },
    "boosting": {
      "median": 0.9,
      "min": 0.71,
      "max": 0.98
    }
  },
  "stability_vs_rule": {
    "model": {
      "wins": 8,
      "ties": 6,
      "losses": 11,
      "median_difference": 0.0
    },
    "boosting": {
      "wins": 5,
      "ties": 4,
      "losses": 16,
      "median_difference": -0.04
    }
  },
  "blind_spot": {
    "pages": 16556,
    "declines": 6746,
    "share_of_all_declines": 0.5303,
    "base_rate": 0.4075,
    "chosen_model_precision_at_50": 0.7,
    "gradient_boosting_preci

### The bottom line — model vs baseline

Every scorer in this notebook on one line, so the week's answer does not have to be reassembled
from four separate tables. Same held-out clients, same metric, same run — nothing here is
recomputed on different data, it is the same numbers gathered in one place.

`blind_spot_p@50` is the second queue from the cell above (the pages the rule scores zero, where
it has no ranking of its own to report). `median_over_25_splits` is the stability check: the
middle result when the split is re-cut 25 times, which is the number to trust over any single
column to its left.

In [16]:
# One table, gathering what the four tables above already measured. No new fits, no new data.
stability_median = {"rule baseline (w04, frozen)": runs.rule.median(),
                    CHOSEN_NAME: runs.model.median(),
                    "gradient boosting": runs.boosting.median()}

final_rows = []
for label, col in SCORERS:
    p50 = precision_at_k_fair(test, col)["tie_fair"]
    final_rows.append({
        "scorer": label,
        "precision@50": p50,
        "vs_baseline_pp": (p50 - BASELINE_P50) * 100,
        "auc": roc_auc_score(test.label_declining, test[col]),
        "imp_protected": int(test.nlargest(TOP_K, col).lost_impressions.sum()),
        "blind_spot_p@50": precision_at_k_fair(blind, col)["tie_fair"] if col in MODELS else np.nan,
        "median_over_25_splits": stability_median.get(label, np.nan),
    })

final_rows.append({"scorer": "picking 50 at random",
                   "precision@50": receipts["random_top50_mean"],
                   "vs_baseline_pp": (receipts["random_top50_mean"] - BASELINE_P50) * 100,
                   "auc": 0.5,
                   "imp_protected": receipts["comparison_imp_protected"]["picking 50 at random (200 draws)"],
                   "blind_spot_p@50": float(np.mean(rand)),
                   "median_over_25_splits": np.nan})

final = pd.DataFrame(final_rows).sort_values("precision@50", ascending=False)
final["vs_baseline_pp"] = final.vs_baseline_pp.round(1)

print(f"MODEL vs BASELINE — held-out clients only: {len(test):,} pages / "
      f"{test.client_hash_id.nunique()} clients | base rate {test.label_declining.mean():.1%}")
print(f"chosen model: {receipts['chosen_model']} {receipts['chosen_params']} on "
      f"{len(FEATURES)} contract features | baseline: w04's frozen rule\n")
print(final.round(3).to_string(index=False))

verdict = ("BEATS" if precision_at_k_fair(test, CHOSEN_NAME)["tie_fair"] > BASELINE_P50
           else "TIES" if precision_at_k_fair(test, CHOSEN_NAME)["tie_fair"] == BASELINE_P50
           else "LOSES TO")
print(f"\nVERDICT: the chosen model {verdict} the frozen baseline at precision@{TOP_K} "
      f"({precision_at_k_fair(test, CHOSEN_NAME)['tie_fair']:.1%} vs {BASELINE_P50:.1%}), "
      f"and protects {int(test.nlargest(TOP_K, 'action_score').lost_impressions.sum()) / max(int(test.nlargest(TOP_K, CHOSEN_NAME).lost_impressions.sum()), 1):.1f}x "
      f"less of the traffic those 50 slots could have protected.")

MODEL vs BASELINE — held-out clients only: 25,771 pages / 9 clients | base rate 49.4%
chosen model: RandomForestClassifier {'n_estimators': 300, 'min_samples_leaf': 400, 'random_state': 42} on 5 contract features | baseline: w04's frozen rule

                        scorer  precision@50  vs_baseline_pp   auc  imp_protected  blind_spot_p@50  median_over_25_splits
steepest slide only (1 column)         0.920             2.0 0.631         117462              NaN                    NaN
             gradient boosting         0.920             2.0 0.666         280650            0.600                   0.90
   rule baseline (w04, frozen)         0.900             0.0 0.627         320709              NaN                   0.94
        random forest (chosen)         0.900             0.0 0.653          53233            0.700                   0.94
       decision tree (depth 3)         0.718           -18.2 0.640          49825            0.484                    NaN
           logistic regr

**And what the errors look like, in four sentences.**

The two scorers fail in opposite directions. All five of the model's wrong picks in its top 50 are
**small pages** (942-2,535 March impressions) that slid hard inside March and then bounced back in
April — one of them more than doubled — while the rule's five wrong picks are **large pages** (up
to 57,720 impressions) that were sliding and then simply held steady, so the model wastes an
editor's afternoon on pages that did not matter and the rule wastes it on pages that turned out
fine. Both are nearly blind to the same thing: of the pages that actually declined, the model
misses **0%** of those already collapsing inside March but **73.7%** of those that were still
growing, which is the decline that arrives with no warning and is exactly the 53% of declines the
rule cannot see either. And the errors are worst where they cost most — the **largest size decile
draws the highest false-alarm rate (52.5%)**, and the model's probabilities run about 20 points
above the outcome (pages scored 0.75 decline 53.6% of the time), so this ships as an ordered queue
and never as a percentage next to a page.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.